# Ch 22. DPO and Modern Alignment — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Compare $\beta=0.01,0.1,1.0$ in the DPO loss and explain the effect.

### Solution

The DPO logit is $\beta[(\log\pi_w-\log\pi_{ref,w})-(\log\pi_l-\log\pi_{ref,l})]$. For a fixed margin, larger $\beta$ saturates the sigmoid, producing sharper preference and potentially smaller effective gradients. The code compares loss and gradient.


In [ ]:
import numpy as np

margin=2.0; results={}
for beta in (.01,.1,1.0):
    logit=beta*margin; loss=float(np.logaddexp(0,-logit)); gradient=float(-beta/(1+np.exp(logit)))
    results[beta]={"loss":loss,"d_loss_d_margin":gradient}
assert results[1.0]["loss"] < results[.1]["loss"] < results[.01]["loss"]
print({str(k):{m:round(v,6) for m,v in r.items()} for k,r in results.items()})


## Problem 2

**Problem**: Explain how the Bradley–Terry model represents a preference pair.

### Solution

Bradley–Terry uses the latent-utility difference as a logit: $P(y_w\succ y_l)=\sigma(r_w-r_l)$. The two order probabilities sum to one, and adding a common constant to both scores changes nothing.


In [ ]:
import numpy as np

chosen=np.array([-1.0,0.0,2.0]); rejected=np.array([0.5,0.0,-1.0]); gaps=chosen-rejected
prob=1/(1+np.exp(-gaps)); reverse=1/(1+np.exp(gaps))
assert np.allclose(prob+reverse,1) and np.isclose(prob[1],.5)
print([{"utility_gap":float(g),"P(chosen preferred)":round(float(p),4)} for g,p in zip(gaps,prob)])


## Problem 3

**Problem**: Compare the memory usage of DPO and PPO.

### Solution

PPO generally needs policy, reference, reward, and value models plus a rollout buffer. DPO centers on chosen/rejected forward passes through policy and reference. Exact bytes depend on optimizer, quantization, and checkpointing, but the lower bound for model states is roughly halved at equal precision.


In [ ]:
# Explicit memory ledger in model-size units; batch buffers are separately measured assumptions.
ppo={"policy":1.0,"reference":1.0,"reward":1.0,"value":1.0,"rollouts":0.30}
dpo={"policy":1.0,"reference":1.0,"paired_batch":0.10}
totals={"PPO":sum(ppo.values()),"DPO":sum(dpo.values())}
assert totals["DPO"] < totals["PPO"] and set(ppo)-set(dpo)=={"reward","value","rollouts"}
print({"components":{"PPO":ppo,"DPO":dpo},"totals":totals,"DPO_fraction":round(totals["DPO"]/totals["PPO"],3)})


## Problem 4

**Problem**: Explain why KTO can train without preference pairs.

### Solution

KTO labels each response desirable or undesirable and optimizes a prospect-theoretic asymmetric loss on a value derived from its log-probability ratio to the reference. Each labeled response yields a loss term, so an explicit same-prompt pair is unnecessary.


In [ ]:
import numpy as np

# Independent labeled examples contribute without constructing chosen/rejected pairs.
log_ratio=np.array([.7,-.4,.2,-.8]); desirable=np.array([1,0,1,0],dtype=bool)
signed_value=np.where(desirable,log_ratio,-log_ratio)
loss=np.logaddexp(0,-signed_value)
assert loss.shape==(4,) and loss[0] < loss[2] and loss[3] < loss[1]
print([{"label":"desirable" if y else "undesirable","log_ratio":float(r),"loss":round(float(l),4)}
       for y,r,l in zip(desirable,log_ratio,loss)])


## Problem 5

**Problem**: Derive the reward–policy relation $r = \beta \log \frac{\pi^*}{\pi_{\text{ref}}} + \beta \log Z$.

### Solution

Solving the KL-regularized objective for each $x$ with a Lagrange multiplier gives $\pi^*(y|x)\propto\pi_{ref}(y|x)e^{r(x,y)/\beta}$. Insert normalizer $Z(x)$, take logs, and rearrange to obtain the relation; $\beta\log Z(x)$ cancels between responses.


In [ ]:
import numpy as np

reward=np.array([1.2,-.3,.7]); beta=.2; reference=np.array([.2,.5,.3])
unnormalized=reference*np.exp(reward/beta); policy=unnormalized/unnormalized.sum(); Z=unnormalized.sum()
reconstructed=beta*np.log(policy/reference)+beta*np.log(Z)
max_error=float(np.max(np.abs(reconstructed-reward)))
assert np.allclose(reconstructed,reward,atol=1e-12) and np.isclose(policy.sum(),1)
print({"reference":reference.tolist(),"optimal_policy":policy.round(6).tolist(),"partition_Z":round(float(Z),6),"max_reconstruction_error":max_error})
